# 09 — Evaluación y comparación

## Objetivo

Evaluar las predicciones OOF con métricas globales, por clase, calibración,
bootstrap, McNemar y **baselines de atajo** (forma de la máscara y contexto).
Todas las cifras de este notebook se leen de los CSV/JSON generados por
`scripts/run_modeling_v2.py`; no se escriben a mano.

## 0. Configuración

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA, V2, REP = ROOT / "data", ROOT / "data" / "v2", ROOT / "reports"
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

In [ ]:
from src.artifacts import ensure_artifacts
ensure_artifacts(
    [DATA / "features_tbx11k_v2.csv"],
    [sys.executable, "scripts/rebuild_features.py", "--workers", "8"],
    "Features v2.",
)
ensure_artifacts(
    [V2 / "summary_models.csv", V2 / "metrics__texture__lr.json",
     V2 / "calibration_lr_texture.json", V2 / "mcnemar_holm.csv",
     V2 / "calibracion_lr_texture.png"],
    [sys.executable, "scripts/run_modeling_v2.py"],
    "Modelado v2 (puede tardar ~20 min).",
)


## 1. Métricas del modelo recomendado

In [ ]:
metrics = json.loads((V2 / "metrics__texture__lr.json").read_text(encoding="utf-8"))
for key in ["metrics_at_0.5", "metrics_youden", "metrics_sens90"]:
    print("\n", key)
    display(pd.Series(metrics[key]).to_frame().T.round(3))

## 2. Comparación de familias (atajo)

In [ ]:
summary = pd.read_csv(V2 / "summary_models.csv")
summary.sort_values("balacc_youden", ascending=False).round(3)

Si `context` o `shape` superan a `texture`, el modelo principal no puede
atribuir todo el rendimiento al pulmón. Esta tabla es el control central del
proyecto.

## 3. Calibración

In [ ]:
cal = json.loads((V2 / "calibration_lr_texture.json").read_text(encoding="utf-8"))
print("Prevalencia real:", round(cal["prevalence"], 4))
print("Brier sin calibrar:", round(cal["brier_raw"], 4))
print("Brier calibrada:", round(cal["brier_calibrated"], 4))
display(Image(filename=str(REP / "v2" / "calibracion_lr_texture.png")))

Las probabilidades de un modelo con `class_weight='balanced'` no están
calibradas a la prevalencia real. La corrección de prior lleva el logit a la
prevalencia observada. Para el CAD se deben usar las probabilidades
calibradas y un umbral fijado por objetivo clínico.

## 4. McNemar con Holm

In [ ]:
mc = pd.read_csv(V2 / "mcnemar_holm.csv")
mc.round(4).head(20)

## 5. Bootstrap estratificado

In [ ]:
ci = metrics["bootstrap_ci_at_0.5"]
pd.DataFrame(ci).T.round(3)

## 6. Comparación con el baseline de fondo

In [ ]:
baselines = summary[summary["feature_set"].isin(["texture", "context", "shape", "shape_quality"])]
baselines[["feature_set", "model", "balacc_youden", "sens_youden", "spec_youden", "ap_youden"]].round(3)

## 7. Recomendación final

- Modelo: **LR sobre textura pulmonar**, con `class_weight='balanced'`.
- Probabilidades: recalibradas a la prevalencia real.
- Umbral: fijado por sensibilidad objetivo (≥ 0.90) en validación anidada.
- Uso: **apoyo al triaje**, no diagnóstico. La sensibilidad y especificidad
  deben validarse en un test externo; TBX11K no tiene IDs de paciente y el
  conjunto `test` (3.302 imágenes) no se usó.
- Limitación principal: la segmentación es un campo pulmonar con margen, no
  parénquima fino; el control de contexto/forma acota el atajo pero no lo
  elimina por completo.

## Conclusiones

El pipeline ahora es coherente: las cifras provienen de los CSV, la
segmentación está validada, el umbral es honesto y los baselines de atajo son
explícitos. La afirmación defendible es: **el modelo detecta TB a partir de
textura del campo pulmonar con rendimiento moderado, y el sesgo de
contexto/fuente está cuantificado**, no que iguale a un radiólogo ni a DL.

**Fin del pipeline de modelado y evaluación.**